# Data Parallelism in the Warehouse: Python UDFs & UDTFs
*Author: Garrett Frere (garrett.frere@snowflake.com)*

**Audience:** folks coming from Spark `applyInPandas` / Pandas UDFs.

| Spark | Snowflake Warehouse | When |
|---|---|---|
| `pandas_udf` (Series -> Series) | **Vectorized UDF** | row-wise transform / feature / scoring, no state across rows |
| `groupBy(k).applyInPandas(fn)` | **Vectorized UDTF** with `end_partition` + `PARTITION BY k` | per-group model fit, stateful logic, "all rows for key k together" |
| broadcast / `mapPartitions` setup | **`@udf_init_once`** | load a model / big lookup **once**, share across workers |

How it runs: Snowflake ships your Python to every node of the warehouse, splits the data into batches (UDF) or partitions (UDTF), and runs them in parallel processes. More nodes = more parallel Python. You write pandas; Snowflake handles the distribution.

**Prereq:** run `udxf_setup.sql` once (creates `STORE_SALES` (100M rows, 1,000 stores x ~100k rows each), `UDXF_STAGE`, `UDXF_S_WH` (Small, 2 nodes), `UDXF_M_WH` (Medium, 4 nodes), `UDXF_L_WH` (Large, 8 nodes)).

**Agenda**
1. Load data as a Snowpark DataFrame
2. Vectorized UDF - inline / anonymous
3. Vectorized UDF - registered (permanent), called from Python **and** SQL
4. `@udf_init_once` - expensive shared initialization
5. Vectorized UDTF - per-store model (the `applyInPandas` replacement)
6. Scaling: Small vs Medium vs Large warehouse

## 1. Setup: session and data
`STORE_SALES` columns: `SALE_ID, STORE_ID, REGION, SKU_ID, SALE_DATE, PRICE, PROMO_FLAG, UNITS`.
Each store has its own hidden price elasticity, so later the UDTF can recover it per store.

In [ ]:
# Session + data as a Snowpark DataFrame (data stays in Snowflake; nothing is pulled locally)
import time
import numpy as np
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.functions import col
from snowflake.snowpark.types import (
    PandasSeriesType, FloatType, LongType, StringType,
    StructType, StructField,
)

session = get_active_session()
session.use_database("ML_PERF")
session.use_schema("PUBLIC")
session.use_warehouse("ML_MODEL_WH")

DEFAULT_WH = "ML_MODEL_WH"
STAGE = "@ML_PERF.PUBLIC.UDXF_STAGE"

sales_df = session.table("ML_PERF.PUBLIC.STORE_SALES")
print(f"rows: {sales_df.count():,}")
sales_df.limit(5).show()

## 2. Vectorized UDF - inline (anonymous / temporary)
The closest thing to a Spark `pandas_udf`. Your function receives a **`pd.Series` per column for a batch of rows** and returns a `pd.Series` of the same length.

- No `name=` -> Snowpark creates a **temporary** function that lives only for this session. Great for exploration.
- `max_batch_size` caps rows per batch (memory control); Snowflake picks batches and spreads them across all nodes/processes.

In [ ]:
# Anonymous vectorized UDF: log-price feature + promo-adjusted revenue
log_price_udf = F.pandas_udf(
    lambda price: np.log(price),
    return_type=PandasSeriesType(FloatType()),
    input_types=[PandasSeriesType(FloatType())],
    max_batch_size=10_000,
)


def net_revenue(price: pd.Series, units: pd.Series, promo: pd.Series) -> pd.Series:
    # 10% discount applied to promo rows
    return price * units * (1 - 0.10 * promo)


net_revenue_udf = F.pandas_udf(
    net_revenue,
    return_type=PandasSeriesType(FloatType()),
    input_types=[PandasSeriesType(FloatType()), PandasSeriesType(LongType()), PandasSeriesType(LongType())],
)

features_df = sales_df.with_columns(
    ["LOG_PRICE", "NET_REVENUE"],
    [log_price_udf(col("PRICE")), net_revenue_udf(col("PRICE"), col("UNITS"), col("PROMO_FLAG"))],
)
features_df.select("STORE_ID", "PRICE", "UNITS", "PROMO_FLAG", "LOG_PRICE", "NET_REVENUE").show(5)

## 3. Vectorized UDF - registered (permanent)
Same Python, but give it a `name`, `is_permanent=True`, and a `stage_location` for the code. Now anyone with the role can call it later - from Python **or** SQL, from tasks, dashboards, dbt, etc.

Using the `@pandas_udf` decorator form here (equivalent to the function-call form above).

In [ ]:
@F.pandas_udf(
    name="ML_PERF.PUBLIC.UDXF_NET_REVENUE",
    is_permanent=True,
    stage_location=STAGE,
    replace=True,
    packages=["pandas", "numpy"],
    return_type=PandasSeriesType(FloatType()),
    input_types=[PandasSeriesType(FloatType()), PandasSeriesType(LongType()), PandasSeriesType(LongType())],
)
def udxf_net_revenue(price: pd.Series, units: pd.Series, promo: pd.Series) -> pd.Series:
    return price * units * (1 - 0.10 * promo)


# Call it from Python by name (e.g. in a different notebook / session later)
rev_by_region = (
    sales_df.with_column("NET_REVENUE", F.call_function("ML_PERF.PUBLIC.UDXF_NET_REVENUE", col("PRICE"), col("UNITS"), col("PROMO_FLAG")))
    .group_by("REGION")
    .agg(F.sum("NET_REVENUE").alias("NET_REVENUE"))
    .sort("REGION")
)
rev_by_region.show()

### Same function, called from SQL
A registered UDF is just a Snowflake function - SQL users get your Python for free.

In [ ]:
%%sql -r sql_rev_by_region
SELECT region,
       SUM(ML_PERF.PUBLIC.UDXF_NET_REVENUE(price, units, promo_flag)) AS net_revenue
FROM ML_PERF.PUBLIC.STORE_SALES
GROUP BY region
ORDER BY region

## 4. `@udf_init_once` - expensive shared initialization
Typical case: a UDF needs a **big model or lookup table**. Without help, each Python worker process on each node loads its own copy, which is slow (paid N times) and memory-hungry (N copies).

`@udf_init_once` runs the decorated function **once per query, in a parent process, before workers are forked**. Workers inherit the result via copy-on-write memory, so it loads once and is held once.

**Rules**
- Zero-arg function; store results in **module globals**.
- **Fork-safe & read-only**: no sessions, network connections, threads, or open file handles; don't mutate the shared object in the handler (mutation = per-worker copy).
- Used in handler **files** registered with `session.udf.register_from_file(...)` - so below we write a small handler module and register it. (This is also the pattern for production code in a repo.)

The example: a per-(store, SKU) price-adjustment matrix with a simulated 5s "model load".

In [ ]:
import os

HANDLER_DIR = "udxf_handlers"
os.makedirs(HANDLER_DIR, exist_ok=True)

handler_src = '''
import time
import numpy as np
import pandas as pd

try:  # server-side implementations
    from _snowflake import udf_init_once, vectorized
except ImportError:  # local testing: same API, no-ops
    from snowflake.snowpark.functions import udf_init_once
    def vectorized(**kwargs):
        return lambda f: f

ADJ = None  # shared, read-only state (filled once, before workers fork)


@udf_init_once
def load_adjustments():
    """Pretend this is loading a large model / reference table from a stage."""
    global ADJ
    time.sleep(5)  # simulated expensive load
    rng = np.random.default_rng(42)
    ADJ = rng.uniform(0.8, 1.2, size=(1001, 501))  # [store_id, sku_id] -> price factor


@vectorized(input=pd.DataFrame)  # handler gets a pandas DataFrame batch; columns 0..3 = args
def adjusted_revenue(df: pd.DataFrame) -> pd.Series:
    store_id, sku_id, price, units = df[0], df[1], df[2], df[3]
    factor = ADJ[store_id.to_numpy(), sku_id.to_numpy()]  # vectorized numpy lookup
    return pd.Series(price.to_numpy() * units.to_numpy() * factor)
'''

handler_path = os.path.join(HANDLER_DIR, "adjusted_revenue.py")
with open(handler_path, "w") as f:
    f.write(handler_src)
print(f"wrote {handler_path}")

In [ ]:
# The handler file declares itself vectorized (@vectorized), so types here are plain SQL types
adjusted_revenue_udf = session.udf.register_from_file(
    file_path=handler_path,
    func_name="adjusted_revenue",
    name="ML_PERF.PUBLIC.UDXF_ADJUSTED_REVENUE",
    is_permanent=True,
    stage_location=STAGE,
    replace=True,
    packages=["pandas", "numpy", "snowflake-snowpark-python"],
    return_type=FloatType(),
    input_types=[LongType(), LongType(), FloatType(), LongType()],
)

t0 = time.time()
adj_by_store = (
    sales_df.with_column("ADJ_REVENUE", adjusted_revenue_udf(col("STORE_ID"), col("SKU_ID"), col("PRICE"), col("UNITS")))
    .group_by("STORE_ID").agg(F.sum("ADJ_REVENUE").alias("ADJ_REVENUE"))
    .sort("STORE_ID")
)
adj_by_store.show(5)
print(f"100M rows scored in {time.time() - t0:.1f}s  (the 5s load ran once per query, not once per worker)")

## 5. Vectorized UDTF - the `applyInPandas` replacement
If your Spark code looks like this:
```python
def fit_store(pdf: pd.DataFrame) -> pd.DataFrame: ...
spark_df.groupBy("store_id").applyInPandas(fit_store, schema="store_id long, elasticity double, ...")
```
the Snowflake equivalent is a **vectorized UDTF**:
- a class with `end_partition(self, df)` - receives **all rows for one partition key** as a pandas DataFrame (columns named by `input_names`)
- declare `input_types=[PandasDataFrameType(...)]` and `output_schema=PandasDataFrameType(...)` - that's what makes it vectorized
- yield a pandas DataFrame matching `output_schema`
- call it with `.over(partition_by="STORE_ID")` - Snowflake routes each store's rows to one process and runs stores in parallel across all nodes

The example: fit a log-log demand model per store, `log(units) = a - elasticity * log(price) + lift * promo`, with 300 bootstrap resamples for a confidence interval (real CPU work, so scaling is visible). The data has a known true elasticity per store, so we can check the answer.

In [ ]:
from snowflake.snowpark.types import PandasDataFrameType

# Vectorized UDTF: input AND output are pandas DataFrames.
# Output key is named STORE (not STORE_ID) so it doesn't collide with the input column.
output_schema = PandasDataFrameType(
    [LongType(), LongType(), FloatType(), FloatType(), FloatType(), FloatType()],
    ["STORE", "N_ROWS", "ELASTICITY", "ELASTICITY_P05", "ELASTICITY_P95", "PROMO_LIFT"],
)

N_BOOT = 300  # bootstrap resamples per store -> real CPU work per partition


class StoreElasticity:
    def end_partition(self, df):
        # Import inside the handler: resolved against the server's numpy version,
        # not pickled from the notebook's (avoids client/server version mismatches)
        import numpy as np
        import pandas as pd

        # df = every row for ONE store (just like the pdf in applyInPandas)
        X = np.column_stack([np.ones(len(df)), np.log(df["PRICE"]), df["PROMO_FLAG"]])
        y = np.log(df["UNITS"].to_numpy())
        beta = np.linalg.lstsq(X, y, rcond=None)[0]

        rng = np.random.default_rng(0)
        boots = []
        for _ in range(N_BOOT):  # bootstrap CI
            idx = rng.integers(0, len(df), len(df))
            boots.append(-np.linalg.lstsq(X[idx], y[idx], rcond=None)[0][1])

        yield pd.DataFrame({
            "STORE": [int(df["STORE_ID"].iloc[0])],
            "N_ROWS": [len(df)],
            "ELASTICITY": [-beta[1]],
            "ELASTICITY_P05": [np.percentile(boots, 5)],
            "ELASTICITY_P95": [np.percentile(boots, 95)],
            "PROMO_LIFT": [beta[2]],
        })


store_elasticity_udtf = session.udtf.register(
    StoreElasticity,
    name="ML_PERF.PUBLIC.UDXF_STORE_ELASTICITY",
    is_permanent=True,
    stage_location=STAGE,
    replace=True,
    packages=["pandas", "numpy"],
    output_schema=output_schema,
    input_types=[PandasDataFrameType([LongType(), FloatType(), LongType(), LongType()])],
    input_names=["STORE_ID", "PRICE", "UNITS", "PROMO_FLAG"],
)


def run_elasticity(source_df=sales_df):
    return source_df.select(
        store_elasticity_udtf(col("STORE_ID"), col("PRICE"), col("UNITS"), col("PROMO_FLAG"))
        .over(partition_by="STORE_ID")
    )


# Same data, limited at call time to 50 stores (~5M rows) so this runs quickly on the X-Small default WH.
# The full 100M-row / 1,000-store run is in the scaling section.
elasticity_df = run_elasticity(sales_df.filter(col("STORE_ID") <= 50))
elasticity_pdf = elasticity_df.sort("STORE").to_pandas()  # 50 rows - safe to pull locally

# Check against the ground truth baked into the synthetic data
elasticity_pdf["TRUE_ELASTICITY"] = 0.5 + (elasticity_pdf["STORE"] * 37 % 100) / 100
print("max |fitted - true|:", (elasticity_pdf["ELASTICITY"] - elasticity_pdf["TRUE_ELASTICITY"]).abs().max().round(4))
elasticity_pdf.head(10)

### Same UDTF from SQL
`PARTITION BY` is the `groupBy` key.

In [ ]:
%%sql -r sql_store_elasticity
SELECT e.*
FROM ML_PERF.PUBLIC.STORE_SALES s,
     TABLE(ML_PERF.PUBLIC.UDXF_STORE_ELASTICITY(s.store_id, s.price, s.units, s.promo_flag)
           OVER (PARTITION BY s.store_id)) e
WHERE s.store_id <= 10   -- limit the call to 10 stores
ORDER BY e.store

## 6. Scaling: same code, bigger warehouse
No code changes - just point the session at a bigger warehouse. Small = 2 nodes, Medium = 4, Large = 8, so each step up doubles the parallel Python processes.

We time both patterns on all 100M rows:
- **UDF** - `UDXF_NET_REVENUE` (section 3): light per-row math, summed by region
- **UDTF** - `store_elasticity_udtf` (section 5): 1,000 stores, 300 bootstrap fits per store

The result cache is disabled so each run really executes; each one gets a small warm-up call first (resume + Python env setup) so the timed run measures the work.

In [ ]:
# UDF scaling: run the vectorized UDF from section 3 on all 100M rows, on each warehouse
from decimal import Decimal, ROUND_HALF_EVEN

session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()  # no result-cache shortcuts

CPUS_PER_NODE = 8
# (warehouse, nodes, credits per hour)
WAREHOUSES = [("UDXF_S_WH", 2, 2), ("UDXF_M_WH", 4, 4), ("UDXF_L_WH", 8, 8)]


def bankers_round(x, places="0.01"):
    # half-to-even rounding to the hundredths (Decimal avoids float tie errors)
    return float(Decimal(str(x)).quantize(Decimal(places), rounding=ROUND_HALF_EVEN))


def run_udf(df=sales_df):
    return (df.with_column("NET_REVENUE", F.call_function("ML_PERF.PUBLIC.UDXF_NET_REVENUE", col("PRICE"), col("UNITS"), col("PROMO_FLAG")))
              .group_by("REGION").agg(F.sum("NET_REVENUE")))


udf_results = []
for wh, nodes, credits_per_hour in WAREHOUSES:
    session.use_warehouse(wh)
    run_udf(sales_df.filter(col("STORE_ID") <= 10)).collect()  # warm-up: wakes the WH + sets up Python
    t0 = time.time()
    run_udf().collect()  # timed: full 100M rows
    secs = time.time() - t0
    query_id = session.sql("SELECT LAST_QUERY_ID()").collect()[0][0]  # look this up in Query History
    udf_results.append({
        "warehouse": wh,
        "nodes": nodes,
        "wh_dop": nodes * CPUS_PER_NODE,  # degree of parallelism = nodes x 8 CPUs
        "seconds": round(secs, 1),
        "credits": bankers_round(secs / 3600 * credits_per_hour),  # runtime x credits/hour
        "query_id": query_id,
    })
    print(udf_results[-1])

session.use_warehouse(DEFAULT_WH)
udf_scaling = pd.DataFrame(udf_results)
udf_scaling

In [ ]:
import matplotlib.pyplot as plt


def plot_scaling(df, name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    labels = df["warehouse"] + "\n(dop " + df["wh_dop"].astype(str) + ")"
    for ax, c, title in [(ax1, "seconds", "Runtime (seconds)"), (ax2, "credits", "Credits used")]:
        bars = ax.bar(labels, df[c], color=["#c6dbef", "#6baed6", "#29B5E8"])
        ax.bar_label(bars, fmt="%.3g")
        ax.set_title(f"{name}: {title}")
    plt.tight_layout()
    plt.show()


plot_scaling(udf_scaling, "UDF")

In [ ]:
# UDTF scaling: run the elasticity UDTF from section 5 on all 100M rows, on each warehouse
udtf_results = []
for wh, nodes, credits_per_hour in WAREHOUSES:
    session.use_warehouse(wh)
    run_elasticity(sales_df.filter(col("STORE_ID") <= 10)).collect()  # warm-up: wakes the WH + sets up Python
    t0 = time.time()
    run_elasticity().collect()  # timed: full 100M rows, 1,000 stores
    secs = time.time() - t0
    query_id = session.sql("SELECT LAST_QUERY_ID()").collect()[0][0]  # look this up in Query History
    udtf_results.append({
        "warehouse": wh,
        "nodes": nodes,
        "wh_dop": nodes * CPUS_PER_NODE,  # degree of parallelism = nodes x 8 CPUs
        "seconds": round(secs, 1),
        "credits": bankers_round(secs / 3600 * credits_per_hour),  # runtime x credits/hour
        "query_id": query_id,
    })
    print(udtf_results[-1])

session.use_warehouse(DEFAULT_WH)
session.sql("ALTER SESSION UNSET USE_CACHED_RESULT").collect()
udtf_scaling = pd.DataFrame(udtf_results)
udtf_scaling

In [ ]:
plot_scaling(udtf_scaling, "UDTF")

### Faster ≠ more expensive
Each warehouse size up doubles both the nodes and the credits per hour (Small = 2, Medium = 4, Large = 8), and billing is per second. So:

**credits = seconds ÷ 3600 × credits per hour**

If each size up halves the runtime, the cost stays about the same: you get the answer faster for the same price.

## Takeaways & tips
- **UDF vs UDTF:** row-independent -> vectorized UDF. Needs "all rows for key k" or state -> vectorized UDTF with `PARTITION BY`.
- **Inline vs registered:** anonymous for exploration; `name` + `is_permanent` + `stage_location` to share and call from SQL.
- **`@udf_init_once`** for heavy models/lookups: loaded once, shared read-only across workers.
- **Import inside UDTF methods** (`import numpy as np` in `end_partition`) to avoid pickling client-side module paths that differ from the server's package versions.
- **Partition key choice:** parallelism is capped by the number of distinct keys. 1,000 stores can use up to 1,000 processes; 3 regions would leave most of a Medium idle. Watch for **skew** (one giant key = one slow process).
- **Memory:** a UDTF partition is held in memory as one pandas DataFrame. Very large keys may need a finer key or a Snowpark-optimized warehouse.
- **`max_batch_size`** on vectorized UDFs bounds per-batch memory.
- **Scaling:** more nodes = more parallel processes. Speedup approaches linear when per-key work dominates; fixed overhead (warehouse resume, env setup, data shuffle) does not scale.
- **Cost:** billing is per second, and each warehouse size doubles both the nodes and the rate. Near-linear speedup means about the same credits for a faster answer. Size up when the work parallelizes well; size down when it doesn't (few keys, heavy skew).